# 15 — Diagnóstico de factibilidad MCR004

**Versión 1.0. Fase 1, paso 4.** Esta libreta examina las veinte rodillas de la regresión histórica ya cerrada como rechazada. Reproduce el método fijado para comprobar igualdad de cajas y píxeles, registrar sus señales internas y mostrar la evidencia de las incidencias a resolución nativa. El informe ayuda a decidir si una corrección futura sería factible; no aprueba el candidato ni modifica las decisiones registradas.

Usa **CPU, Colab 2026.07 y Python 3.12**, y el mismo secreto `KNEE_DATA_ROOT` de las libretas anteriores. Puedes ejecutar las celdas en orden o usar «Ejecutar todo», atendiendo la autorización de acceso a Drive. No hay formularios ni casillas de confirmación y no debes diagnosticar marcas.

La salida nueva se guarda en Drive bajo la raíz privada autorizada. Si ya existe, se detiene sin sobrescribirla: conserva esa salida y comparte el mensaje para revisar su estado. No borres carpetas ni vuelvas a ejecutar las libretas 13 o 14 para resolverlo. Las imágenes y los informes individuales son privados; comparte únicamente el resumen público final.


## 1. Conectar Drive y obtener el código

Se crea una copia temporal nueva del repositorio y se registra su revisión exacta. Se usan los archivos históricos existentes; no se seleccionan participantes nuevos.


In [ ]:
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, HTML
import json, os, re, subprocess, sys, tempfile

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Selecciona Colab 2026.07 / Python 3.12 antes de ejecutar esta libreta.')

drive.mount('/content/drive')
diag_private_value = userdata.get('KNEE_DATA_ROOT')
if not diag_private_value:
    raise RuntimeError('Falta el secreto KNEE_DATA_ROOT de las libretas anteriores.')
diag_private_root = Path(diag_private_value).resolve()
if not diag_private_root.is_dir():
    raise FileNotFoundError('La raíz privada autorizada no está disponible.')
os.environ['KNEE_DATA_ROOT'] = str(diag_private_root)
os.environ['KNEE_DIAGNOSTIC_EXECUTION_ENVIRONMENT'] = 'Google Colab, libreta 15'
diag_completed = False
diag_tests_passed = False

def diag_run_checked(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd,
                            text=True, capture_output=True)
    if result.returncode:
        print('Código de salida:', result.returncode)
        print('SALIDA ESTÁNDAR:\n' + (result.stdout or '(vacía)'))
        print('ERROR INTERNO:\n' + (result.stderr or '(vacío)'))
        raise RuntimeError(
            'El diagnóstico se detuvo. Conserva el mensaje y cualquier salida existente; '
            'no borres archivos ni repitas los cierres anteriores.')
    return result.stdout

diag_task_dir = Path(tempfile.mkdtemp(prefix='mcr004_diagnostico_', dir='/content'))
diag_repo_dir = diag_task_dir/'Knee'
diag_branch = 'codex/fase-0-base-reproducible'
diag_run_checked(['git', 'clone', '--depth', '1', '--branch', diag_branch,
    'https://github.com/AlonsoYess/Knee.git', diag_repo_dir])
diag_git_commit = diag_run_checked(['git', 'rev-parse', 'HEAD'], cwd=diag_repo_dir).strip()
if not re.fullmatch(r'[0-9a-f]{40}', diag_git_commit):
    raise RuntimeError('No se obtuvo una revisión válida del código.')
if diag_run_checked(['git', 'symbolic-ref', '--short', 'HEAD'], cwd=diag_repo_dir).strip() != diag_branch:
    raise RuntimeError('La rama descargada no coincide con la rama autorizada.')
if diag_run_checked(['git', 'status', '--porcelain'], cwd=diag_repo_dir).strip():
    raise RuntimeError('La copia recién descargada contiene cambios inesperados.')
diag_required_files = (
    'src/knee/roi_mcr004_diagnostic.py',
    'configs/roi_mcr004_historical.example.json',
    'requirements-mcr004.txt',
    'tests/test_roi_mcr004_diagnostic.py',
)
if not all((diag_repo_dir/relative).is_file() for relative in diag_required_files):
    raise RuntimeError('La rama no contiene la entrega diagnóstica completa. Comparte este mensaje.')
diag_config_path = diag_repo_dir/'configs'/'roi_mcr004_historical.example.json'
diag_config = json.loads(diag_config_path.read_text(encoding='utf-8'))
diag_historical_dir = (diag_private_root/diag_config['output_dir']).resolve()
if not diag_historical_dir.is_relative_to(diag_private_root) or not diag_historical_dir.is_dir():
    raise RuntimeError('No está disponible la salida histórica bajo la raíz autorizada.')
diag_output_relative = 'outputs/auditorias/fase_1/paso_4_localizacion_tibiofemoral/mcr004_diagnostico_v1'
diag_output_dir = (diag_private_root/diag_output_relative).resolve()
if not diag_output_dir.is_relative_to(diag_private_root) or diag_output_dir == diag_historical_dir:
    raise RuntimeError('La salida diagnóstica no es una carpeta privada separada válida.')
if diag_output_dir.exists():
    raise FileExistsError(
        'La salida diagnóstica ya existe. Se conserva intacta; comparte este mensaje '
        'para comprobar si la ejecución anterior terminó. No la borres ni cambies su nombre.')
print('Código del diagnóstico:', diag_git_commit)
print('Salida separada de la regresión histórica: preparada para una ejecución nueva.')


## 2. Preparar el entorno aislado y comprobar las pruebas

Se utiliza un entorno nuevo con las dependencias fijadas de MCR004. Las pruebas sintéticas verifican el programa, no la calidad anatómica de las radiografías.


In [ ]:
diag_tests_passed = False
diag_venv_dir = diag_task_dir/'venv_diagnostic'
diag_run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==21.7.11'])
diag_run_checked([sys.executable, '-m', 'virtualenv', '--no-periodic-update', diag_venv_dir])
diag_python = diag_venv_dir/'bin'/'python'
diag_run_checked([diag_python, '-m', 'pip', 'install', '-q',
    '-r', diag_repo_dir/'requirements-mcr004.txt'])
diag_run_checked([diag_python, '-m', 'pip', 'install', '-q', '--no-deps', '-e', diag_repo_dir])
diag_version_code = "import json,sys;from importlib.metadata import version;print(json.dumps({'python':sys.version.split()[0],**{p:version(p) for p in ('numpy','scipy','opencv-python','pydicom','Pillow')}}))"
diag_environment = json.loads(diag_run_checked([diag_python, '-c', diag_version_code]))
diag_expected = {'numpy':'1.26.4','scipy':'1.14.1','opencv-python':'4.10.0.84',
                 'pydicom':'3.0.1','Pillow':'11.3.0'}
if not diag_environment['python'].startswith('3.12.') or any(
    diag_environment[key] != value for key, value in diag_expected.items()):
    raise RuntimeError('El entorno no coincide con las versiones fijadas. No se ejecuta el diagnóstico.')
diag_run_checked([diag_python, '-m', 'unittest', 'discover', '-s', 'tests',
    '-p', 'test_roi_mcr004*.py', '-v'], cwd=diag_repo_dir)
diag_tests_passed = True
print('Entorno aislado:', diag_environment)
print('Pruebas técnicas sintéticas: correctas. No acreditan calidad anatómica.')


## 3. Ejecutar únicamente el diagnóstico histórico

El programa verifica el cierre rechazado y las huellas antes de reproducir las mismas veinte candidatas. Exige igualdad exacta con los recortes ya guardados y conserva el cierre anterior. Si alguna comprobación falla, muestra el error completo y detiene la ejecución. No crea recortes de producción ni selecciona una muestra nueva.


In [ ]:
diag_completed = False
if not diag_tests_passed:
    raise RuntimeError('Faltan las pruebas técnicas; no se ejecuta el diagnóstico.')
if diag_output_dir.exists():
    raise FileExistsError(
        'La salida diagnóstica ya existe. No se sobrescribe; conserva los archivos '
        'y comparte el mensaje para revisar el estado de esa ejecución.')
diag_raw_summary = diag_run_checked([
    diag_python, '-m', 'knee.roi_mcr004_diagnostic',
    '--config', diag_config_path,
    '--output-dir', diag_output_relative,
    '--git-commit', diag_git_commit,
], cwd=diag_repo_dir)
diag_summary = json.loads(diag_raw_summary)
diag_expected_outputs = (
    'diagnostico_privado.json',
    'galeria_diagnostica.html',
    'resumen_diagnostico_publico.json',
    'registro_diagnostico.json',
)
if not all((diag_output_dir/name).is_file() for name in diag_expected_outputs):
    raise RuntimeError('La salida está incompleta. Consérvala y comparte el error; no repitas ni borres archivos.')
diag_saved_summary = json.loads(
    (diag_output_dir/'resumen_diagnostico_publico.json').read_text(encoding='utf-8'))
if diag_saved_summary != diag_summary:
    raise RuntimeError('El resumen guardado no coincide con la salida del programa. Conserva ambos para revisión.')
diag_completed = True
print(json.dumps(diag_summary, ensure_ascii=False, indent=2))
print('Diagnóstico guardado. Paso 4: abierto; el candidato mantiene su rechazo.')


## 4. Ver la evidencia guardada y compartir el resumen

La galería incluye las veinte correspondencias y las vistas diagnósticas con su contexto. Las imágenes se muestran a tamaño natural dentro de paneles desplazables; usa las barras para ver zonas que excedan el ancho de la pantalla. No necesitas asignar nuevas decisiones ni interpretar clínicamente las marcas.

Los archivos quedaron en la carpeta privada de diagnóstico. Comparte el JSON público de la celda anterior. Codex podrá revisar los artefactos privados disponibles para preparar la conclusión técnica; la generación de este informe no modifica umbrales ni habilita etapas posteriores.


In [ ]:
if not diag_completed:
    raise RuntimeError('No hay un diagnóstico completo y verificado en esta ejecución.')
diag_gallery_path = (diag_output_dir/'galeria_diagnostica.html').resolve()
if not diag_gallery_path.is_relative_to(diag_output_dir) or not diag_gallery_path.is_file():
    raise RuntimeError('La galería privada no está disponible dentro de la salida verificada.')
display(HTML(diag_gallery_path.read_text(encoding='utf-8')))
print('La galería y los informes individuales son privados. Comparte únicamente el resumen público.')
